In [1]:
from julia.api import Julia
from julia import Main

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from multiprocessing import Pool
from concurrent.futures import ProcessPoolExecutor
import concurrent.futures

import scipy.special as sp
import os

import pathlib
from pathlib import Path
import json

#fitters

import pybobyqa
import time
import cma
import csv

def to_float64(df):
    num_cols = df.select_dtypes(include=["number"]).columns
    df[num_cols] = df[num_cols].astype("float64")
    return df

Which Fit?

In [2]:
fit_name = "BroadBump42LogGaussAlpha1NoLambda2"

approximate_total_xsec = True
data_uncertainty_only = False  # use experimental + PDF covariance in chi2

N_replicas = 100
vary_data_points = True
use_pdf_shift = True
pdf_shift_mode = "sequential"  # "random" or "sequential"

output_csv_name = "replica_0324.csv"
append_to_existing_csv = True
use_random_seed = True
replica_seed = 12345


Read Files

In [3]:
TMD_fitting_root = "../"
def include(name):
    path = os.path.join(TMD_fitting_root, name)
    Main.eval(f'include(raw"{path}")')

include(f"Cards/{fit_name}.jl")
include(f"DY/DY_table_{Main.flavor_scheme}.jl")

# Data
file_root = f"../Data/{Main.data_name}/Cutted/DY"
matrix_root = f"../Data/{Main.data_name}/Covariance_Matrices/DY"
table_root = f"../Tables/{Main.table_name}/DY"
total_root = f"../Data/DY_total_xsec/{Main.pdf_name}"
error_sets_root = f"../Data/PDF_Matrices/{Main.error_sets_name}/DY"
pdf_diff_root = f"../Data/PDF_Differences/{Main.error_sets_name}"

initial_params = Main.initial_params


By file or by experiment?

In [4]:
data_selections = "by_experiment"  # "by_file" or "by_experiment"

In [5]:
experiments =[
    'ATLAS_7',
    'ATLAS_8', 
    #'ATLAS_13', 
    'CDF_I',
    'CDF_II',
    'CMS_7',
    'CMS_8',
    'CMS_13',    
    'D0_I',
    'D0_II',
    'D0_II_mu',
    'E288',
    'E605',
    'E772',
    'LHCb_7',
    'LHCb_8',
    'LHCb_13',    
    #'PHENIX',
    'STAR'
]



#["E288","E605","E772","ATLAS", "CMS", "D0", "CDF", "LHCb", "PHENIX", "STAR"]

if data_selections == "by_file":
    file_names = [

    #----------------------------------------------------------------------------
    # ATLAS
    #----------------------------------------------------------------------------

    #"ATLAS/ATLAS_7TeV_y_0_1.csv",
    #"ATLAS/ATLAS_7TeV_y_1_2.csv",
    #"ATLAS/ATLAS_7TeV_y_2_2.4.csv",

    #"ATLAS/ATLAS_8TeV_Q_44_66.csv",
    #"ATLAS/ATLAS_8TeV_Q_116_150.csv",

    #"ATLAS/ATLAS_8TeV_y_0_0.4.csv"
    #"ATLAS/ATLAS_8TeV_y_0.4_0.8.csv"
    #"ATLAS/ATLAS_8TeV_y_0.8_1.2.csv"
    #"ATLAS/ATLAS_8TeV_y_1.2_1.6.csv",
    #"ATLAS/ATLAS_8TeV_y_1.6_2.csv",
    #"ATLAS/ATLAS_8TeV_y_2_2.4.csv",

    #----------------------------------------------------------------------------
    # CDF
    #----------------------------------------------------------------------------

    #"CDF/CDF_RunI.csv",
    #"CDF/CDF_RunII.csv",

    #----------------------------------------------------------------------------
    # CMS
    #----------------------------------------------------------------------------

    #"CMS/CMS_7TeV.csv",
    #"CMS/CMS_8TeV.csv",
    
    #"CMS/CMS_13TeV_y_0_0.4.csv",
    #"CMS/CMS_13TeV_y_0.4_0.8.csv",
    #"CMS/CMS_13TeV_y_0.8_1.2.csv",
    #"CMS/CMS_13TeV_y_1.2_1.6.csv",
    #"CMS/CMS_13TeV_y_1.6_2.4.csv",

    #----------------------------------------------------------------------------
    # D0
    #----------------------------------------------------------------------------

    #"D0/D0_RunI.csv",
    #"D0/D0_RunII.csv",
    #"D0/D0_RunIImu.csv",

    #----------------------------------------------------------------------------
    # LHCb
    #----------------------------------------------------------------------------

    #"LHCb/LHCb_7TeV.csv",
    #"LHCb/LHCb_8TeV.csv",
    #"LHCb/LHCb_13TeV.csv",

    #----------------------------------------------------------------------------
    # Phenix
    #----------------------------------------------------------------------------

    #"PHENIX/PHENIX_200.csv",

    #----------------------------------------------------------------------------
    # STAR
    #----------------------------------------------------------------------------

    #"STAR/STAR_510.csv",

    #----------------------------------------------------------------------------
    # E288
    #----------------------------------------------------------------------------

    #"E288/E288_200_Q_4_5.csv",
    #"E288/E288_200_Q_5_6.csv",
    #"E288/E288_200_Q_6_7.csv",
    #"E288/E288_200_Q_7_8.csv",
    #"E288/E288_200_Q_8_9.csv",
    #"E288/E288_200_Q_10_11.csv",

    #"E288/E288_300_Q_4_5.csv",
    #"E288/E288_300_Q_5_6.csv",
    #"E288/E288_300_Q_6_7.csv",
    #"E288/E288_300_Q_7_8.csv",
    #"E288/E288_300_Q_8_9.csv",
    #"E288/E288_300_Q_10_11.csv",
    #"E288/E288_300_Q_11_12.csv",

    #"E288/E288_400_Q_5_6.csv",
    #"E288/E288_400_Q_6_7.csv",
    #"E288/E288_400_Q_7_8.csv",
    #"E288/E288_400_Q_8_9.csv",
    #"E288/E288_400_Q_10_11.csv",
    #"E288/E288_400_Q_11_12.csv",
    #"E288/E288_400_Q_12_13.csv",
    #"E288/E288_400_Q_13_14.csv",

    #----------------------------------------------------------------------------
    # E605
    #----------------------------------------------------------------------------

    #"E605/E605_Q_7_8.csv",
    #"E605/E605_Q_8_9.csv",
    #"E605/E605_Q_10.5_11.5.csv",
    #"E605/E605_Q_11.5_13.5.csv",
    #"E605/E605_Q_13.5_18.csv",

    #----------------------------------------------------------------------------
    # E772
    #----------------------------------------------------------------------------

    #"E772/E772_Q_5_6.csv",
    #"E772/E772_Q_6_7.csv",
    #"E772/E772_Q_7_8.csv",
    #"E772/E772_Q_8_9.csv",
    #"E772/E772_Q_11_12.csv",
    #"E772/E772_Q_12_13.csv",
    #"E772/E772_Q_13_14.csv",
    #"E772/E772_Q_14_15.csv",
    ]

In [6]:
from pathlib import Path

file_excludes = [
    "E772/E772-5Q6.csv",
    "E772/E772-6Q7.csv",
    "E772/E772-7Q8.csv",
    "E772/E772-8Q9.csv",
]

if data_selections == "by_experiment":
    file_names = []
    for experiment in experiments:
        exp_dir = Path(file_root) / experiment
        for p in sorted(exp_dir.glob("*.csv")):
            if str(experiment + "/" + p.name) in file_excludes:
                continue
            file_names.append(str(Path(experiment) / p.name)) 

display(file_names)

['ATLAS_7\\ATLAS7-00y10.csv',
 'ATLAS_7\\ATLAS7-10y20.csv',
 'ATLAS_7\\ATLAS7-20y24.csv',
 'ATLAS_8\\ATLAS8-00y04.csv',
 'ATLAS_8\\ATLAS8-04y08.csv',
 'ATLAS_8\\ATLAS8-08y12.csv',
 'ATLAS_8\\ATLAS8-116Q150.csv',
 'ATLAS_8\\ATLAS8-12y16.csv',
 'ATLAS_8\\ATLAS8-16y20.csv',
 'ATLAS_8\\ATLAS8-20y24.csv',
 'ATLAS_8\\ATLAS8-46Q66.csv',
 'CDF_I\\CDF1.csv',
 'CDF_II\\CDF2.csv',
 'CMS_7\\CMS7.csv',
 'CMS_8\\CMS8.csv',
 'CMS_13\\CMS13-00y04.csv',
 'CMS_13\\CMS13-04y08.csv',
 'CMS_13\\CMS13-08y12.csv',
 'CMS_13\\CMS13-106Q170.csv',
 'CMS_13\\CMS13-12y16.csv',
 'CMS_13\\CMS13-16y24.csv',
 'CMS_13\\CMS13-170Q350.csv',
 'CMS_13\\CMS13-350Q1000.csv',
 'D0_I\\D01.csv',
 'D0_II\\D02.csv',
 'D0_II_mu\\D02mu.csv',
 'E288\\E228-200-4Q5.csv',
 'E288\\E228-200-5Q6.csv',
 'E288\\E228-200-6Q7.csv',
 'E288\\E228-200-7Q8.csv',
 'E288\\E228-200-8Q9.csv',
 'E288\\E228-300-11Q12.csv',
 'E288\\E228-300-4Q5.csv',
 'E288\\E228-300-5Q6.csv',
 'E288\\E228-300-6Q7.csv',
 'E288\\E228-300-7Q8.csv',
 'E288\\E228-300-8Q9.cs

Read Data

In [7]:
data_list = dict()
matrix_data_list = dict()
matrix_total_list = dict()
df_total_xsec = to_float64(pd.read_csv(f"{total_root}.csv"))
total_xsec_names = df_total_xsec['name'].tolist()

for file in tqdm(file_names):

    df_data = to_float64(pd.read_csv(f"{file_root}/{file}"))
    data_list[file] = df_data
    
    matrix_data = to_float64(pd.read_csv(f"{matrix_root}/{file}"))
    
    if data_uncertainty_only == True:
        matrix_total = matrix_data
    else:
        matrix_PDF = to_float64(pd.read_csv(f"{error_sets_root}/{file}"))
        matrix_total = matrix_data + matrix_PDF
    
    matrix_data_list[file] = matrix_data
    matrix_total_list[file] = matrix_total

    name_short = Path(file).stem
    if name_short in total_xsec_names:
        total_xsec = df_total_xsec[df_total_xsec['name'] == name_short]["total_xsec"].values[0]
        data_list[file]['total_xsec'] = total_xsec*np.ones(len(data_list[file]))
        print(f"{name_short}'s total xsec = {total_xsec} added")

  9%|▉         | 5/57 [00:00<00:01, 44.79it/s]

ATLAS7-00y10's total xsec = 1.0 added
ATLAS7-10y20's total xsec = 1.0 added
ATLAS7-20y24's total xsec = 1.0 added


 33%|███▎      | 19/57 [00:00<00:01, 37.43it/s]

CMS7's total xsec = 1.0 added
CMS8's total xsec = 1.0 added


 60%|█████▉    | 34/57 [00:00<00:00, 41.45it/s]

D02's total xsec = 1.0 added
D02mu's total xsec = 1.0 added


100%|██████████| 57/57 [00:01<00:00, 41.66it/s]


Prediction

In [8]:
Params = Main.Params_Struct(*[np.float32(x) for x in initial_params]) 
#Main.set_params(Main.VRAM, Params) 

for i in range(10):
    Params = Main.Params_Struct(*[np.float32(x) for x in initial_params])                  
    predictions,t = Main.xsec_dict(Main.rel_paths, Main.VRAM)
    print(round(t*1000,2), "ms")

303.1 ms
37.37 ms
37.24 ms
37.09 ms
37.21 ms
37.14 ms
37.1 ms
36.82 ms
37.05 ms
37.18 ms


In [9]:
def get_file_length():

    file_lengths = dict()

    for file in file_names:

        df = to_float64(pd.read_csv(f"{file_root}/{file}"))

        file_lengths[file] = len(df)

    return file_lengths

file_lengths = get_file_length()

In [10]:
def _norm(p: str) -> str:
    return os.path.normpath(p).replace('\\', '/')

df_total_xsec = to_float64(pd.read_csv(f"{total_root}.csv"))
total_xsec_names = df_total_xsec['name'].tolist()

def prediction_reformat(predictions):
    preds = {_norm(k): v for k, v in predictions.items()}  # normalize keys once
    df_predictions = {}

    for file in file_names:
        n = file_lengths[file]
        base = os.path.splitext(file)[0]
        xs = []
        for i in range(n):
            table_path = _norm(os.path.join(table_root, f"{base}/{i}.jls"))
            xs.append(preds[table_path])
        df_predictions[file] = np.array(xs)

        if approximate_total_xsec == True and Path(file).stem in total_xsec_names:
            data_xsec = data_list[file]["xsec"].to_numpy()
            dσ = sum(data_xsec)/sum(xs)
            df_predictions[file] = np.array(dσ * np.array(xs))
            #display(file, 1/dσ)

    return df_predictions

df_predictions = prediction_reformat(predictions)

In [11]:
import pickle

def clone_data_list(source):
    return {file: df.copy(deep=True) for file, df in source.items()}

nominal_data_list = clone_data_list(data_list)

def _pdf_dataset_key(file):
    return str(Path(file).with_suffix("")).replace("\\", "/")

def load_pdf_shift_replica(path):
    with path.open("rb") as f:
        pdf_diff_dict = pickle.load(f)

    pdf_shift_list = {}
    for file in file_names:
        prefix = f"{_pdf_dataset_key(file)}/"
        values = []
        for i in range(file_lengths[file]):
            key = f"{prefix}{i}"
            if key not in pdf_diff_dict:
                raise KeyError(f"Missing PDF-difference entry: {key}")
            values.append(pdf_diff_dict[key])
        pdf_shift_list[file] = np.asarray(values, dtype=np.float64)

    return pdf_shift_list

zero_pdf_shift_list = {
    file: np.zeros(file_lengths[file], dtype=np.float64)
    for file in file_names
}

pdf_shift_replicas = {}
pdf_replica_ids = np.array([], dtype=int)
if use_pdf_shift:
    pdf_diff_paths = sorted(Path(pdf_diff_root).glob("*.pkl"), key=lambda p: int(p.stem))
    if not pdf_diff_paths:
        raise FileNotFoundError(f"No PDF-difference replicas found under {pdf_diff_root}")

    pdf_shift_replicas = {
        int(path.stem): load_pdf_shift_replica(path)
        for path in tqdm(pdf_diff_paths, desc="Loading PDF-difference replicas")
    }
    pdf_replica_ids = np.array(sorted(pdf_shift_replicas), dtype=int)
    print(f"Loaded {len(pdf_replica_ids)} PDF-difference replicas from {pdf_diff_root}")
else:
    print("PDF-difference replica shifts are disabled.")

current_pdf_replica_id = None
current_pdf_shift_list = zero_pdf_shift_list

def apply_pdf_shift(df_predictions, pdf_shift_list):
    shifted = {}
    for file in file_names:
        shifted[file] = np.asarray(df_predictions[file], dtype=np.float64) + pdf_shift_list[file]
    return shifted


Loading PDF-difference replicas: 100%|██████████| 100/100 [00:00<00:00, 101.90it/s]

Loaded 100 PDF-difference replicas from ../Data/PDF_Differences/MSHT20N3LO-MC


Chi2

In [12]:
ASWZ_b_array = np.linspace(0.12,0.78,12)*5.067731
ASWZ_prediction = np.array([
    -0.08158508158508182,
    -0.1701631701631705,
    -0.2400932400932403,
    -0.34265734265734293,
    -0.37062937062937085,
    -0.4265734265734267,
    -0.4498834498834501,
    -0.44522144522144536,
    -0.4965034965034967,
    -0.5710955710955714,
    -0.6363636363636365,
    -0.7016317016317017
    ])
ASWZ_upper = np.array([
    0.18414918414918402,
    0.11421911421911402,
    0.09557109557109533,
    0.002331002331002141,
    0.016317016317016098,
    -0.034965034965035224,
    -0.034965034965035224,
    -0.011655011655011815,
    -0.034965034965035224,
    -0.05361305361305391,
    -0.05827505827505863,
    -0.04895104895104918
    ])
ASWZ_error = np.array(ASWZ_upper) - np.array(ASWZ_prediction)

def chi2_lattice(): 
    CS_list = []
    for b in ASWZ_b_array :
        Q = 2.0
        CS = Main.CS_total_func(b, Q)
        CS_list.append(CS)
    chi2dN = np.sum( (CS_list - ASWZ_prediction)**2 / ASWZ_error**2 ) / len(ASWZ_b_array)
    return chi2dN

def timed(func):
    t0 = time.perf_counter()
    out = func()
    return out, time.perf_counter() - t0

#chi2dN, t = timed(chi2_lattice)
#print("χ^2/N from LATTICE =", chi2dN, ", took", round(t, 4), "seconds")

In [13]:
def get_chi2dN(df_predictions):

    N_list = dict()
    chi2dN_list = dict()
    chi2_total = 0.0
    N_total = 0

    for file in file_names:

        data_xsec = data_list[file]["xsec"].to_numpy()
        pred_xsec = df_predictions[file]
        diff_xsec = data_xsec - pred_xsec

        covariance_matrix_inv = np.linalg.inv(matrix_total_list[file])

        N = len(data_xsec)

        chi2 = diff_xsec @ covariance_matrix_inv @ diff_xsec

        chi2_total += chi2
        N_total += N
        chi2dN_list[file] = float(round(chi2/N, 3))
        N_list[file] = N

    chi2dN = chi2_total / N_total
    return chi2dN, chi2dN_list, N_list

chi2dN, chi2dN_list, N_list = get_chi2dN(df_predictions)

print(f"Total χ^2/N = {chi2dN:.3f}")
display(chi2dN_list)

Total χ^2/N = 0.991


{'ATLAS_7\\ATLAS7-00y10.csv': 0.625,
 'ATLAS_7\\ATLAS7-10y20.csv': 3.394,
 'ATLAS_7\\ATLAS7-20y24.csv': 1.222,
 'ATLAS_8\\ATLAS8-00y04.csv': 2.847,
 'ATLAS_8\\ATLAS8-04y08.csv': 1.025,
 'ATLAS_8\\ATLAS8-08y12.csv': 0.354,
 'ATLAS_8\\ATLAS8-116Q150.csv': 0.551,
 'ATLAS_8\\ATLAS8-12y16.csv': 2.22,
 'ATLAS_8\\ATLAS8-16y20.csv': 1.011,
 'ATLAS_8\\ATLAS8-20y24.csv': 0.966,
 'ATLAS_8\\ATLAS8-46Q66.csv': 0.911,
 'CDF_I\\CDF1.csv': 0.543,
 'CDF_II\\CDF2.csv': 0.738,
 'CMS_7\\CMS7.csv': 1.445,
 'CMS_8\\CMS8.csv': 0.753,
 'CMS_13\\CMS13-00y04.csv': 1.299,
 'CMS_13\\CMS13-04y08.csv': 0.828,
 'CMS_13\\CMS13-08y12.csv': 0.436,
 'CMS_13\\CMS13-106Q170.csv': 0.818,
 'CMS_13\\CMS13-12y16.csv': 0.215,
 'CMS_13\\CMS13-16y24.csv': 0.222,
 'CMS_13\\CMS13-170Q350.csv': 1.087,
 'CMS_13\\CMS13-350Q1000.csv': 1.55,
 'D0_I\\D01.csv': 0.576,
 'D0_II\\D02.csv': 0.795,
 'D0_II_mu\\D02mu.csv': 0.542,
 'E288\\E228-200-4Q5.csv': 0.299,
 'E288\\E228-200-5Q6.csv': 0.27,
 'E288\\E228-200-6Q7.csv': 0.463,
 'E288\\E228-2

Objective

In [14]:
def objective(params):
    try:
        params_cl = Main.Params_Struct(*[np.float32(x) for x in params])
        Main.set_params(Main.VRAM, params_cl)

        predictions, t = Main.xsec_dict(Main.rel_paths, Main.VRAM)

        df_predictions = prediction_reformat(predictions)
        df_predictions = apply_pdf_shift(df_predictions, current_pdf_shift_list)
        chi2dN, _, _ = get_chi2dN(df_predictions)

        if not np.isfinite(chi2dN):
            return 1e5
        return chi2dN

    except Exception as e:
        return 1e5

print(objective(initial_params))

# freeze parameters

initial_params = np.asarray(initial_params, float)
frozen_idx = np.asarray(Main.frozen_indices, int)

dim_full = len(initial_params)
mask = np.ones(dim_full, dtype=bool)
mask[frozen_idx] = False
free_idx = np.where(mask)[0]   # indices that WILL be fitted

frozen_vals = initial_params[frozen_idx].copy()

def objective_with_freeze(params_free):

    params_free = np.asarray(params_free, float)
    full = initial_params.copy()
    full[free_idx] = params_free
    full[frozen_idx] = frozen_vals
    return objective(full)

print(objective_with_freeze(initial_params[free_idx]))


0.9905143951392045
0.990512158063254


Bounds

In [15]:
bounds_raw = np.asarray(Main.bounds_raw, float)[free_idx]

lower_bounds, upper_bounds = np.array(list(zip(*bounds_raw)))

def objective_normalized(params):

    normalized_params = lower_bounds + params * (upper_bounds - lower_bounds)

    return objective_with_freeze(normalized_params)

def objective_log(params):
    return np.log10(objective_normalized(params))

def normalize_params(params):
    return (params - lower_bounds) / (upper_bounds - lower_bounds)

def denormalize_params(params):
    return lower_bounds + params * (upper_bounds - lower_bounds)

theta0 = normalize_params(np.array(initial_params)[free_idx])
dim = len(bounds_raw)

## Replica Refit Workflow

Each fit replica draws:

1. A pseudo-dataset from the experimental covariance matrix.
2. A random PDF-difference replica from `Data/PDF_Differences/{error_sets_name}`.

The chosen PDF-difference vector is added pointwise to the theory prediction before evaluating `χ²`.


In [16]:
if use_random_seed:
    replica_rng = np.random.default_rng()
else:
    replica_rng = np.random.default_rng(replica_seed)

randomize_start = False
alpha = 9.0

local_stage1_maxfun = 220
local_stage2_maxfun = 120
local_stage1_rhobeg = 0.07
local_stage1_rhoend = 5e-5
local_stage2_rhobeg = 0.03
local_stage2_rhoend = 1e-6

rescue_stage1_maxfun = 180
rescue_stage2_maxfun = 120
rescue_stage1_rhobeg = 0.09
rescue_stage1_rhoend = 1e-4
rescue_stage2_rhobeg = 0.04
rescue_stage2_rhoend = 1e-6

base_jitter_sigma = 0.05
previous_jitter_sigma = 0.03

results_path = Path("replica_data/", output_csv_name)

param_columns = [f"param_{i}" for i in range(dim_full)]


In [17]:
from types import SimpleNamespace

def generate_experimental_replica_data(rng):
    replica_data = clone_data_list(nominal_data_list)
    if not vary_data_points:
        return replica_data

    for file in file_names:
        xsecs = nominal_data_list[file]["xsec"].to_numpy(np.float64)
        cov = np.asarray(matrix_data_list[file], np.float64)
        cov = 0.5 * (cov + cov.T)

        eps = np.finfo(cov.dtype).eps * max(1.0, float(np.max(np.diag(cov))))
        L = np.linalg.cholesky(cov + eps * np.eye(len(xsecs)))

        replica_data[file]["xsec"] = xsecs + L @ rng.standard_normal(len(xsecs))

    return replica_data

def sample_pdf_replica(rng, replica_id):
    if not use_pdf_shift:
        return None, zero_pdf_shift_list

    if pdf_shift_mode == "random":
        pdf_replica_id = int(rng.choice(pdf_replica_ids))
    elif pdf_shift_mode == "sequential":
        pdf_replica_id = int(replica_id) + 1
        if pdf_replica_id not in pdf_shift_replicas:
            raise KeyError(
                f"Sequential PDF-shift mode requires PDF replica {pdf_replica_id} for fit replica {replica_id}, "
                f"but it was not found under {pdf_diff_root}"
            )
    else:
        raise ValueError(
            f"Unknown pdf_shift_mode={pdf_shift_mode!r}; use 'random' or 'sequential'"
        )

    return pdf_replica_id, pdf_shift_replicas[pdf_replica_id]

def draw_theta0(rng):
    if not randomize_start:
        return theta0.copy()

    while True:
        cand = rng.beta(alpha, alpha, size=dim)
        val = float(objective_log(cand))
        if val < 1.0:
            return cand

def jitter_theta(theta, sigma, rng):
    return np.clip(theta + rng.normal(scale=sigma, size=dim), 0.0, 1.0)

def dedupe_theta_starts(starts, atol=1e-10):
    unique = []
    for theta in starts:
        if theta is None:
            continue
        if not any(np.allclose(theta, prev, atol=atol, rtol=0.0) for prev in unique):
            unique.append(theta.copy())
    return unique

def build_replica_starts(rng):
    base_theta = draw_theta0(rng)
    starts = [
        base_theta,
        jitter_theta(base_theta, base_jitter_sigma, rng),
        jitter_theta(base_theta, previous_jitter_sigma, rng),
    ]
    return dedupe_theta_starts(starts)[:3]

def fallback_result(theta_start, exc):
    return SimpleNamespace(
        x=np.clip(theta_start.copy(), 0.0, 1.0),
        f=float(objective_log(theta_start)),
        nf=0,
        flag=-999,
        message=str(exc),
    )

def solve_replica_stage(theta_start, *, maxfun, rhobeg, rhoend, seek_global_minimum):
    try:
        return pybobyqa.solve(
            objective_log,
            np.clip(theta_start, 0.0, 1.0),
            bounds=(np.zeros(dim), np.ones(dim)),
            maxfun=maxfun,
            rhobeg=rhobeg,
            rhoend=rhoend,
            scaling_within_bounds=True,
            seek_global_minimum=seek_global_minimum,
        )
    except Exception as exc:
        return fallback_result(theta_start, exc)

def better_result(res_a, res_b):
    if res_a is None:
        return res_b
    if res_b is None:
        return res_a
    return res_b if float(res_b.f) < float(res_a.f) else res_a

def run_local_track(theta_start):
    res1 = solve_replica_stage(
        theta_start,
        maxfun=local_stage1_maxfun,
        rhobeg=local_stage1_rhobeg,
        rhoend=local_stage1_rhoend,
        seek_global_minimum=False,
    )
    res2 = solve_replica_stage(
        np.clip(res1.x, 0.0, 1.0),
        maxfun=local_stage2_maxfun,
        rhobeg=local_stage2_rhobeg,
        rhoend=local_stage2_rhoend,
        seek_global_minimum=False,
    )
    return better_result(res1, res2), int(res1.nf) + int(res2.nf)

def run_rescue_track(theta_start):
    res1 = solve_replica_stage(
        theta_start,
        maxfun=rescue_stage1_maxfun,
        rhobeg=rescue_stage1_rhobeg,
        rhoend=rescue_stage1_rhoend,
        seek_global_minimum=True,
    )
    res2 = solve_replica_stage(
        np.clip(res1.x, 0.0, 1.0),
        maxfun=rescue_stage2_maxfun,
        rhobeg=rescue_stage2_rhobeg,
        rhoend=rescue_stage2_rhoend,
        seek_global_minimum=False,
    )
    return better_result(res1, res2), int(res1.nf) + int(res2.nf)

def fit_replica(rng):
    starts = build_replica_starts(rng)
    best_res = None
    best_label = None
    total_nfev = 0

    for idx, theta_start in enumerate(starts):
        res, stage_nfev = run_local_track(theta_start)
        total_nfev += stage_nfev
        if (best_res is None) or (float(res.f) < float(best_res.f)):
            best_res = res
            best_label = f"local-{idx}"

    rescue_starts = dedupe_theta_starts([
        np.clip(best_res.x, 0.0, 1.0),
        jitter_theta(np.clip(best_res.x, 0.0, 1.0), base_jitter_sigma, rng),
    ])

    for idx, theta_start in enumerate(rescue_starts):
        res, stage_nfev = run_rescue_track(theta_start)
        total_nfev += stage_nfev
        if float(res.f) < float(best_res.f):
            best_res = res
            best_label = f"rescue-{idx}"

    return best_res, total_nfev, best_label

def prepare_results_file(path):
    header = ["replica_id", "pdf_replica_id", "success", "nfev", "best_chi2dN", *param_columns]
    with path.open("w", newline="") as f:
        csv.writer(f).writerow(header)

def get_start_replica_id(path):
    if (not append_to_existing_csv) or (not path.exists()) or path.stat().st_size == 0:
        return 0

    with path.open("r", newline="") as f:
        rows = list(csv.DictReader(f))

    if not rows:
        return 0

    return max(int(row["replica_id"]) for row in rows) + 1

if (not append_to_existing_csv) or (not results_path.exists()) or results_path.stat().st_size == 0:
    prepare_results_file(results_path)

start_replica_id = get_start_replica_id(results_path)

if use_pdf_shift and pdf_shift_mode == "sequential":
    required_pdf_replica_ids = range(start_replica_id + 1, start_replica_id + N_replicas + 1)
    missing_pdf_replica_ids = [pdf_id for pdf_id in required_pdf_replica_ids if pdf_id not in pdf_shift_replicas]
    if missing_pdf_replica_ids:
        preview = missing_pdf_replica_ids[:10]
        suffix = "..." if len(missing_pdf_replica_ids) > 10 else ""
        raise ValueError(
            f"Sequential PDF-shift mode requires PDF replicas {start_replica_id + 1} through {start_replica_id + N_replicas}, "
            f"but these IDs are missing under {pdf_diff_root}: {preview}{suffix}"
        )


In [18]:
replica_results = []
replica_ids = range(start_replica_id, start_replica_id + N_replicas)

for replica_id in tqdm(replica_ids, desc="Replica refits"):
    data_list = generate_experimental_replica_data(replica_rng)
    current_pdf_replica_id, current_pdf_shift_list = sample_pdf_replica(replica_rng, replica_id)
    res, total_nfev, best_label = fit_replica(replica_rng)

    best_chi2dN = float(10 ** res.f)
    optimal_params_free = denormalize_params(res.x)

    full_params = initial_params.copy()
    full_params[free_idx] = optimal_params_free
    full_params[frozen_idx] = frozen_vals

    row = {
        "replica_id": replica_id,
        "pdf_replica_id": current_pdf_replica_id,
        "success": int(res.flag == 0),
        "nfev": int(total_nfev),
        "best_chi2dN": best_chi2dN,
    }
    for name, value in zip(param_columns, full_params):
        row[name] = float(value)

    replica_results.append(row)

    fmt = lambda x: f"{float(x):.8g}"
    with results_path.open("a", newline="") as f:
        csv.writer(f).writerow(
            [
                row["replica_id"],
                row["pdf_replica_id"],
                row["success"],
                row["nfev"],
                fmt(row["best_chi2dN"]),
                *[fmt(row[name]) for name in param_columns],
            ]
        )

    pdf_label = current_pdf_replica_id if current_pdf_replica_id is not None else "off"
    print(
        f"Replica {replica_id}: pdf={pdf_label}, "
        f"chi2/N={best_chi2dN:.3f}, nfev={int(total_nfev)}, track={best_label}"
    )

data_list = clone_data_list(nominal_data_list)
current_pdf_replica_id = None
current_pdf_shift_list = zero_pdf_shift_list

replica_results_df = pd.read_csv(results_path)


Replica refits:   1%|          | 1/100 [02:02<3:22:49, 122.92s/it]

Replica 0: pdf=1, chi2/N=1.761, nfev=1590, track=rescue-0


Replica refits:   2%|▏         | 2/100 [04:03<3:18:02, 121.25s/it]

Replica 1: pdf=2, chi2/N=1.823, nfev=1568, track=rescue-0


Replica refits:   3%|▎         | 3/100 [05:55<3:09:38, 117.31s/it]

...

Replica 97: pdf=98, chi2/N=1.886, nfev=1538, track=rescue-0


Replica refits:  99%|█████████▉| 99/100 [3:14:12<01:55, 115.39s/it]

Replica 98: pdf=99, chi2/N=1.924, nfev=1518, track=rescue-0


Replica refits: 100%|██████████| 100/100 [3:16:10<00:00, 117.70s/it]

Replica 99: pdf=100, chi2/N=1.967, nfev=1575, track=rescue-1


In [19]:
display(replica_results_df.head())

print(f"Wrote {len(replica_results_df)} replica refits to {results_path}")
print(f"Replica ids: {start_replica_id} to {start_replica_id + len(replica_results_df) - 1}")
print()
print(replica_results_df[["best_chi2dN", "nfev"]].describe())

if use_pdf_shift:
    print()
    print(f"PDF shift mode: {pdf_shift_mode}")
    print("PDF replica usage:")
    display(replica_results_df["pdf_replica_id"].value_counts().sort_index().rename("count").to_frame())
else:
    print()
    print("PDF replica shifts were disabled for this run.")


,replica_id,pdf_replica_id,success,nfev,best_chi2dN,param_0,param_1,param_2,param_3,param_4,param_5,param_6,param_7,param_8
0,0,1,1,1590,1.760621,-0.000808,1.067443,-2.158223,-5.090047,1.059374,-0.550237,1.435322,0.065957,0.033349
1,1,2,0,1568,1.822936,0.014953,0.788787,-1.902008,-5.223488,0.639931,-0.830473,1.589615,0.074648,0.020001
2,2,3,1,1474,1.871218,0.041030,0.960542,-1.956359,-5.005228,0.771849,-0.255985,1.535313,0.072112,0.025215
3,3,4,0,1620,2.039987,-0.057762,0.894539,-1.988473,-5.335862,1.384307,-0.561117,1.213681,0.070745,0.044346
4,4,5,0,1620,1.960847,-0.024462,1.052912,-2.294709,-5.194345,1.303810,-0.631446,1.329398,0.068660,0.037772


Wrote 100 replica refits to replica_data\replica_0324.csv
Replica ids: 0 to 99

       best_chi2dN         nfev
count   100.000000   100.000000
mean      1.922545  1577.610000
std       0.153556    36.620327
min       1.582315  1409.000000
25%       1.823925  1560.000000
50%       1.914958  1582.000000
75%       2.013894  1602.250000
max       2.550069  1620.000000

PDF shift mode: sequential
PDF replica usage:


,count
pdf_replica_id,
1,1
2,1
3,1
4,1
5,1
...,...
96,1
97,1
98,1
